# Experiment 41: Exact 33B + CatBoost Ensemble

Test the exact 33B digit decomposition pipeline with proper 5-fold OOF predictions, then blend it with the strong Experiment 39 CatBoost model.

No submission CSV or OOF prediction CSV is generated.

## 1. Imports and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

TRAIN_PATH = '../data/train.csv'
TARGET = 'Will_Buy_EV'

train = pd.read_csv(TRAIN_PATH)

y = train[TARGET].map({'No': 0, 'Yes': 1}).astype(np.int8)
X = train.drop(columns=[TARGET, 'id']).copy()

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

print('Rows:', len(X))
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

Rows: 668665
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 2. Exact 33B Identity Encoding

In [2]:
def make_identity_key(series):
    return series.astype('string').fillna('__MISSING__')


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        'value': values,
        'target': target.to_numpy()
    })

    global_mean = float(target.mean())

    stats = (
        temp.groupby('value', dropna=False)['target']
        .agg(['mean', 'count'])
    )

    smoothed = (
        stats['count'] * stats['mean']
        + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return (
        values.map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def add_identity_features(X_fit, y_fit, X_apply, columns, n_splits=3, smoothing=20):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf_inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:
        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf_inner.split(X_fit, y_fit):
            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f'{col}__identity_target'] = oof_values

        X_apply[f'{col}__identity_target'] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        frequencies = fit_keys.value_counts(dropna=False)

        X_fit[f'{col}__identity_frequency'] = (
            fit_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

        X_apply[f'{col}__identity_frequency'] = (
            apply_keys.map(frequencies)
            .fillna(0)
            .astype(float)
            .to_numpy()
        )

    return X_fit, X_apply

## 3. Exact 33B Digit Decomposition

In [3]:
def add_digit_features(X_frame, columns):
    X_frame = X_frame.copy()

    for col in columns:
        values = pd.to_numeric(X_frame[col], errors='coerce')

        integer_values = values.abs().round()

        X_frame[f'{col}__digits'] = (
            np.floor(np.log10(integer_values.clip(lower=1))) + 1
        )

        divisor = 10 ** (X_frame[f'{col}__digits'] - 1)

        X_frame[f'{col}__first_digit'] = (
            integer_values / divisor
        ).fillna(0).astype(float)

        X_frame[f'{col}__first_digit'] = np.floor(
            X_frame[f'{col}__first_digit']
        )

        X_frame[f'{col}__last_digit'] = (
            integer_values.fillna(0).astype(np.int64) % 10
        )

        def digit_sum(v):
            if pd.isna(v):
                return np.nan
            s = str(int(abs(v)))
            return sum(int(ch) for ch in s)

        X_frame[f'{col}__digit_sum'] = integer_values.map(digit_sum)

        X_frame[f'{col}__parity'] = (
            integer_values.fillna(0).astype(np.int64) % 2
        )

        X_frame[f'{col}__mod100'] = (
            integer_values.fillna(0).astype(np.int64) % 100
        )

        X_frame[f'{col}__mod1000'] = (
            integer_values.fillna(0).astype(np.int64) % 1000
        )

        X_frame[f'{col}__ends_zero'] = (
            integer_values.fillna(0).astype(np.int64) % 10 == 0
        ).astype(np.int8)

    return X_frame

## 4. Exact 33B XGBoost Configuration

In [4]:
def build_preprocessor(X_frame):
    numeric = X_frame.select_dtypes(include=['number']).columns.tolist()
    categorical = X_frame.select_dtypes(exclude=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer([
        ('num', numeric_pipeline, numeric),
        ('cat', categorical_pipeline, categorical)
    ])


def run_33b_fold(X_tr, y_tr, X_va, y_va, fold):
    print('')
    print('-' * 60)
    print(f'33B Fold {fold}')
    print('-' * 60)

    X_tr_id, X_va_id = add_identity_features(
        X_tr,
        y_tr,
        X_va,
        numeric_cols,
        n_splits=3,
        smoothing=20
    )

    X_tr_digit = add_digit_features(X_tr_id, numeric_cols)
    X_va_digit = add_digit_features(X_va_id, numeric_cols)

    preprocessor = build_preprocessor(X_tr_digit)

    print('Encoding...')

    X_tr_encoded = preprocessor.fit_transform(X_tr_digit)
    X_va_encoded = preprocessor.transform(X_va_digit)

    print('Encoded shape:', X_tr_encoded.shape)

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr_encoded,
        y_tr,
        eval_set=[(X_va_encoded, y_va)],
        verbose=False
    )

    pred = model.predict_proba(X_va_encoded)[:, 1]
    score = roc_auc_score(y_va, pred)

    print(f'33B Fold {fold} AUC: {score:.6f}')

    return pred, score

## 5. Exact 33B 5-Fold OOF

In [5]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_oof = np.zeros(len(X), dtype=float)
xgb_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y.iloc[tr_idx]
    y_va = y.iloc[va_idx]

    pred, score = run_33b_fold(
        X_tr, y_tr, X_va, y_va, fold
    )

    xgb_oof[va_idx] = pred
    xgb_scores.append(score)

xgb_score = roc_auc_score(y, xgb_oof)

print('')
print('=' * 70)
print('41A EXACT 33B OOF RESULT')
print('=' * 70)
print(f'33B 5-fold OOF ROC-AUC: {xgb_score:.6f}')
print(f'Original 33B benchmark: 0.945331')
print(f'Difference: {xgb_score - 0.945331:+.6f}')


------------------------------------------------------------
33B Fold 1
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B Fold 1 AUC: 0.944663

------------------------------------------------------------
33B Fold 2
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B Fold 2 AUC: 0.945209

------------------------------------------------------------
33B Fold 3
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B Fold 3 AUC: 0.946415

------------------------------------------------------------
33B Fold 4
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B Fold 4 AUC: 0.945843

------------------------------------------------------------
33B Fold 5
------------------------------------------------------------
Encoding...
Encoded shape: (534932, 94)
33B Fold 5 AUC: 0.945617

41A 

## 6. Experiment 39 CatBoost OOF

In [6]:
cat_oof = np.zeros(len(X), dtype=float)
cat_scores = []

cat_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

subsidy = X['Subsidy_Available'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)
home = X['Home_Charging_Possible'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)

X_cat = X.copy()
X_cat['Subsidy_x_EnvConcern'] = subsidy * X_cat['Environmental_Concern_Level']
X_cat['Subsidy_x_Income'] = subsidy * X_cat['Annual_Income_USD']
X_cat['Subsidy_x_HomeCharging'] = subsidy * home

value_identity_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]

for col in value_identity_cols:
    X_cat[f'{col}__value_id'] = X_cat[col].astype('string').fillna('__MISSING__')

cat_identity_cols = cat_cols + [
    f'{c}__value_id' for c in value_identity_cols
]

for col in cat_identity_cols:
    X_cat[col] = X_cat[col].astype('string').fillna('__MISSING__')

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cat, y), start=1):
    X_tr = X_cat.iloc[tr_idx].copy()
    X_va = X_cat.iloc[va_idx].copy()
    y_tr = y.iloc[tr_idx]
    y_va = y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function='Logloss',
        eval_metric='AUC',
        l2_leaf_reg=3,
        random_strength=1,
        bootstrap_type='Bayesian',
        bagging_temperature=1,
        random_seed=7,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=cat_identity_cols,
        eval_set=(X_va, y_va),
        use_best_model=True,
        verbose=False
    )

    pred = model.predict_proba(X_va)[:, 1]
    cat_oof[va_idx] = pred

    score = roc_auc_score(y_va, pred)
    cat_scores.append(score)

    print(f'CatBoost Fold {fold} AUC: {score:.6f}')

cat_score = roc_auc_score(y, cat_oof)

print('')
print(f'41B CatBoost OOF: {cat_score:.6f}')

CatBoost Fold 1 AUC: 0.944431
CatBoost Fold 2 AUC: 0.944978
CatBoost Fold 3 AUC: 0.946433
CatBoost Fold 4 AUC: 0.945627
CatBoost Fold 5 AUC: 0.945427

41B CatBoost OOF: 0.945370


## 7. Raw Probability Blend Search

In [7]:
blend_results = []

for cat_weight in np.arange(0.0, 1.001, 0.025):
    blend = (
        cat_weight * cat_oof
        + (1.0 - cat_weight) * xgb_oof
    )

    auc = roc_auc_score(y, blend)
    blend_results.append((auc, cat_weight))

blend_results.sort(reverse=True)

print('Top raw probability blends:')

for auc, cat_weight in blend_results[:15]:
    print(
        f'CatBoost {cat_weight:.3f} | '
        f'33B {1-cat_weight:.3f} | '
        f'AUC {auc:.6f}'
    )

best_raw_auc, best_raw_weight = blend_results[0]

Top raw probability blends:
CatBoost 0.400 | 33B 0.600 | AUC 0.945694
CatBoost 0.425 | 33B 0.575 | AUC 0.945693
CatBoost 0.375 | 33B 0.625 | AUC 0.945693
CatBoost 0.450 | 33B 0.550 | AUC 0.945692
CatBoost 0.350 | 33B 0.650 | AUC 0.945691
CatBoost 0.475 | 33B 0.525 | AUC 0.945690
CatBoost 0.325 | 33B 0.675 | AUC 0.945688
CatBoost 0.500 | 33B 0.500 | AUC 0.945686
CatBoost 0.300 | 33B 0.700 | AUC 0.945684
CatBoost 0.525 | 33B 0.475 | AUC 0.945681
CatBoost 0.275 | 33B 0.725 | AUC 0.945678
CatBoost 0.550 | 33B 0.450 | AUC 0.945675
CatBoost 0.250 | 33B 0.750 | AUC 0.945672
CatBoost 0.575 | 33B 0.425 | AUC 0.945668
CatBoost 0.225 | 33B 0.775 | AUC 0.945664


## 8. Rank Blend Search

In [8]:
cat_rank = pd.Series(cat_oof).rank(method='average', pct=True).to_numpy()
xgb_rank = pd.Series(xgb_oof).rank(method='average', pct=True).to_numpy()

rank_results = []

for cat_weight in np.arange(0.0, 1.001, 0.025):
    blend = (
        cat_weight * cat_rank
        + (1.0 - cat_weight) * xgb_rank
    )

    auc = roc_auc_score(y, blend)
    rank_results.append((auc, cat_weight))

rank_results.sort(reverse=True)

print('Top rank blends:')

for auc, cat_weight in rank_results[:15]:
    print(
        f'CatBoost {cat_weight:.3f} | '
        f'33B {1-cat_weight:.3f} | '
        f'AUC {auc:.6f}'
    )

best_rank_auc, best_rank_weight = rank_results[0]

Top rank blends:
CatBoost 0.425 | 33B 0.575 | AUC 0.945724
CatBoost 0.400 | 33B 0.600 | AUC 0.945723
CatBoost 0.450 | 33B 0.550 | AUC 0.945723
CatBoost 0.375 | 33B 0.625 | AUC 0.945721
CatBoost 0.475 | 33B 0.525 | AUC 0.945721
CatBoost 0.350 | 33B 0.650 | AUC 0.945718
CatBoost 0.500 | 33B 0.500 | AUC 0.945718
CatBoost 0.325 | 33B 0.675 | AUC 0.945714
CatBoost 0.525 | 33B 0.475 | AUC 0.945714
CatBoost 0.300 | 33B 0.700 | AUC 0.945708
CatBoost 0.550 | 33B 0.450 | AUC 0.945708
CatBoost 0.275 | 33B 0.725 | AUC 0.945701
CatBoost 0.575 | 33B 0.425 | AUC 0.945701
CatBoost 0.250 | 33B 0.750 | AUC 0.945693
CatBoost 0.600 | 33B 0.400 | AUC 0.945692


## 9. Final Experiment 41 Results

In [9]:
results = pd.DataFrame([
    {'Experiment': '41A_Exact_33B_OOF', 'ROC_AUC': xgb_score},
    {'Experiment': '41B_CatBoost_Seed7_OOF', 'ROC_AUC': cat_score},
    {'Experiment': '41C_Raw_Probability_Blend', 'ROC_AUC': best_raw_auc},
    {'Experiment': '41D_Rank_Blend', 'ROC_AUC': best_rank_auc}
])

results = results.sort_values(
    'ROC_AUC',
    ascending=False
).reset_index(drop=True)

print('')
print('=' * 70)
print('EXPERIMENT 41 RESULTS')
print('=' * 70)
print(results.to_string(index=False))

best_score = float(results.iloc[0]['ROC_AUC'])
best_name = results.iloc[0]['Experiment']

print('')
print(f'BEST: {best_name}')
print(f'BEST OOF ROC-AUC: {best_score:.6f}')
print(f'vs 33B: {best_score - 0.945331:+.6f}')
print(f'vs Exp39: {best_score - 0.945426:+.6f}')
print(f'vs 0.946000: {best_score - 0.946000:+.6f}')
print(f'vs 0.950000: {best_score - 0.950000:+.6f}')

print('')
print('No submission generated.')
print('No OOF prediction CSV generated.')
print('Experiment 41 complete.')


EXPERIMENT 41 RESULTS
               Experiment  ROC_AUC
           41D_Rank_Blend 0.945724
41C_Raw_Probability_Blend 0.945694
        41A_Exact_33B_OOF 0.945541
   41B_CatBoost_Seed7_OOF 0.945370

BEST: 41D_Rank_Blend
BEST OOF ROC-AUC: 0.945724
vs 33B: +0.000393
vs Exp39: +0.000298
vs 0.946000: -0.000276
vs 0.950000: -0.004276

No submission generated.
No OOF prediction CSV generated.
Experiment 41 complete.
